# Step 13 — the figures, two tier

Reads `step11_panels.rds` and `step12_clusters.rds`. Writes the PNGs and `step13_zoom_depth.rds`.

Panels are 490–863 probes, so per-protein labels are not legible at full width. Rather than
silently subsetting:

- **Tier 1, overview** — the whole panel, both dendrograms, no protein labels. Row dendrogram and
  row names share the **left** side; column dendrogram and column names share the **top**.
- **Tier 2, labelled zooms** — descend the column dendrogram until every cluster fits the label
  budget, then draw one labelled heatmap per cluster, and report how many levels that took.

`MAX_LABELS = 80` is a rendering limit, not a statistical one, and is stated as such.

One trap, documented in `src/final_matrix.R`: with `column_split` set, `Heatmap()`'s `column_title`
becomes the **per-slice** title. Figure captions go on `draw()`.

In [ ]:
suppressMessages({library(ComplexHeatmap); library(circlize); library(grid)})
source("../src/paths.R")
options(stringsAsFactors = FALSE); set.seed(42)
P <- readRDS(art("step11_panels.rds")); K <- readRDS(art("step12_clusters.rds"))
D <- P$D; spec <- P$spec; MINK_P <- P$MINK_P; LINK <- P$LINK; SITES <- P$SITES
RES <- K$RES
W <- lapply(SITES, function(s) readRDS(art("wgcna_%s.rds", s))); names(W) <- SITES

MAX_LABELS <- 80      # a RENDERING limit, not a statistical one

suppressMessages({library(ComplexHeatmap);library(circlize);library(grid)})
dmink <- function(m) dist(m, method="minkowski", p=MINK_P)

ann_of <- function(s, rn, traits){
  m <- W[[s]]$meta[rn, , drop=FALSE]
  cols <- intersect(spec$source_column[match(traits, spec$name)], names(m))
  m[, cols, drop=FALSE]
}

# ---- TIER 1: overview, no protein labels, both dendrograms, names on same side
tier1 <- function(nm, level=2){          # level 2 -> the 4-group cut
  r <- RES[[nm]]; d <- r$d; Z <- r$Z
  K <- 2^level                           # split IS the dendrogram cut, so pass k
  Heatmap(Z, name="z-score", col=colorRamp2(c(-2,0,2),c("#2166AC","white","#B2182B")),
    row_split=K, column_split=K,
    cluster_rows=r$hclust$hr, cluster_columns=r$hclust$hc,
    cluster_row_slices=FALSE, cluster_column_slices=FALSE,
    show_row_dend=TRUE,  row_dend_side="left",  row_dend_width=unit(14,"mm"),
    show_row_names=TRUE, row_names_side="left", row_names_gp=gpar(fontsize=3),
    show_column_dend=TRUE, column_dend_side="top", column_dend_height=unit(14,"mm"),
    show_column_names=FALSE,
    row_title_gp=gpar(fontsize=8), column_title_gp=gpar(fontsize=8),
    left_annotation=rowAnnotation(df=ann_of(d$cohort, rownames(Z), d$traits),
                                  annotation_name_gp=gpar(fontsize=5), show_legend=FALSE),
    heatmap_legend_param=list(labels_gp=gpar(fontsize=7), title_gp=gpar(fontsize=8)))
}
for (cond in c("all15","varsel","union")){
  nms <- paste(SITES, cond)
  png(art("fig13_overview_%s.png", cond), width=3600, height=1700, res=150)
  grid.newpage(); pushViewport(viewport(layout=grid.layout(1,3)))
  for (i in seq_along(nms)){
    pushViewport(viewport(layout.pos.row=1, layout.pos.col=i))
    d <- D[[nms[i]]]
    draw(tier1(nms[i]), newpage=FALSE,
      column_title=sprintf("cohort %s -- %s: %d modules, %s", d$cohort, cond,
                           length(d$keep), size_str(d$sel)),
      column_title_gp=gpar(fontsize=10, fontface="bold"))
    popViewport()
  }
  popViewport(); invisible(dev.off())
  cat("wrote", basename(art("fig13_overview_%s.png", cond)), "\n")
}

# ---- TIER 2: descend the column dendrogram until clusters fit MAX_LABELS
depth_for <- function(hc, n, maxlab){
  for (lev in 1:8){ k <- 2^lev
    if (k > n) return(list(level=lev, k=n))
    if (max(table(cutree(hc,k))) <= maxlab) return(list(level=lev, k=k)) }
  list(level=8, k=256)
}
zoom_report <- NULL
for (s in SITES){
  nm <- paste(s,"all15"); r <- RES[[nm]]; Z <- r$Z
  dd <- depth_for(r$hclust$hc, ncol(Z), MAX_LABELS)
  cl <- cutree(r$hclust$hc, dd$k)
  zoom_report <- rbind(zoom_report, data.frame(cohort=s, panel=ncol(Z),
    levels_down=dd$level, clusters=dd$k, largest=max(table(cl))))
  ord <- order(table(cl), decreasing=TRUE)
  keep <- as.integer(names(sort(table(cl), decreasing=TRUE)))[1:min(6,dd$k)]
  png(art("fig13_zoom_%s.png", s), width=3400, height=2000, res=150)
  grid.newpage(); pushViewport(viewport(layout=grid.layout(2,3)))
  for (i in seq_along(keep)){
    g <- names(cl)[cl==keep[i]]
    pushViewport(viewport(layout.pos.row=(i-1)%/%3+1, layout.pos.col=(i-1)%%3+1))
    ht <- Heatmap(Z[,g,drop=FALSE], name="z-score",
      col=colorRamp2(c(-2,0,2),c("#2166AC","white","#B2182B")),
      row_split=4, cluster_rows=r$hclust$hr, cluster_row_slices=FALSE,
      cluster_columns=TRUE, clustering_distance_columns=dmink, clustering_method_columns=LINK,
      show_row_dend=TRUE, row_dend_side="left", row_dend_width=unit(8,"mm"),
      show_row_names=FALSE,
      show_column_dend=TRUE, column_dend_side="top", column_dend_height=unit(12,"mm"),
      show_column_names=TRUE, column_names_side="top", column_names_gp=gpar(fontsize=5),
      row_title_gp=gpar(fontsize=7), show_heatmap_legend=(i==1),
      heatmap_legend_param=list(labels_gp=gpar(fontsize=6), title_gp=gpar(fontsize=7)))
    draw(ht, newpage=FALSE, column_title=sprintf("%s cluster %d -- %s", s, keep[i], size_str(g)),
         column_title_gp=gpar(fontsize=9, fontface="bold"))
    popViewport()
  }
  popViewport(); invisible(dev.off())
  cat("wrote", basename(art("fig13_zoom_%s.png", s)), "\n")
}
cat("\nlabelled-zoom depth (limit", MAX_LABELS, "proteins per panel):\n")
print(zoom_report, row.names=FALSE)

saveRDS(zoom_report, art("step13_zoom_depth.rds"))


Reaching 80 proteins per panel takes **five levels** in A and C and **four** in B. A 4-way cut
— one level then one more — leaves roughly 190 proteins per panel, so the analysis cut and the
legible cut are different depths, and both are produced.